# 🌊 YOLO-UDD v2.0 - Underwater Debris Detection

[![GitHub](https://img.shields.io/badge/GitHub-YOLO--UDD--v2.0-blue)](https://github.com/kshitijkhede/YOLO-UDD-v2.0)
[![Paper](https://img.shields.io/badge/Paper-arXiv-red)](https://arxiv.org)

Novel architecture with **TAFM (Turbidity-Adaptive Fusion Module)** for underwater debris detection.

**Features:**
- 🌊 Turbidity-adaptive feature fusion
- 🎯 COCO-style mAP evaluation
- 🚀 Ready for deployment

---

## 📥 Step 1: Clone Repository

In [ ]:
# Clone the repository
!git clone https://github.com/kshitijkhede/YOLO-UDD-v2.0.git
%cd YOLO-UDD-v2.0

print("✓ Repository cloned successfully!")

## 📦 Step 2: Install Dependencies

In [ ]:
# Install required packages
!pip install -q torch torchvision torchaudio
!pip install -q albumentations opencv-python-headless
!pip install -q tqdm pyyaml tensorboard

print("✓ Dependencies installed!")

## 💾 Step 3: Mount Google Drive & Setup Dataset

**Important:** Upload your TrashCAN dataset to Google Drive first!

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("✓ Google Drive mounted!")
print("\n📁 Please ensure your dataset is uploaded to Google Drive")
print("   Expected structure: instance_version/")
print("   - instances_train_trashcan.json")
print("   - instances_val_trashcan.json")
print("   - train/ (folder with images)")
print("   - val/ (folder with images)")

## ⚙️ Step 4: Configure Training Settings

In [ ]:
import os

# =============================================================================
# CONFIGURATION - UPDATE THESE VALUES
# =============================================================================

# Dataset path - UPDATE THIS to your Google Drive location!
DATASET_PATH = '/content/drive/MyDrive/trashcan_dataset/instance_version'

# Training hyperparameters
BATCH_SIZE = 8      # Reduce to 4 if you get OOM (Out of Memory) errors
EPOCHS = 50         # Increase to 100-300 for full training
LEARNING_RATE = 0.01
SAVE_DIR = 'runs/train'

# =============================================================================
# Verify Configuration
# =============================================================================

print("="*70)
print("⚙️  Training Configuration")
print("="*70)
print(f"Dataset Path: {DATASET_PATH}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"Save Directory: {SAVE_DIR}")
print("="*70)

## 🔍 Step 5: Pre-flight Checks

In [ ]:
import torch
import os

print("="*70)
print("🔍 Pre-flight Checks")
print("="*70)

# Check GPU
print(f"\n✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print("\n🚀 GPU detected! Training will be fast.")
else:
    print("\n⚠️  WARNING: No GPU detected!")
    print("   Training will be VERY slow on CPU.")
    print("   Enable GPU: Runtime → Change runtime type → T4 GPU")

# Check dataset
print(f"\n📁 Checking dataset at: {DATASET_PATH}")

if os.path.exists(DATASET_PATH):
    train_json = os.path.join(DATASET_PATH, 'instances_train_trashcan.json')
    val_json = os.path.join(DATASET_PATH, 'instances_val_trashcan.json')
    train_dir = os.path.join(DATASET_PATH, 'train')
    val_dir = os.path.join(DATASET_PATH, 'val')
    
    print("✓ Dataset path exists")
    print(f"  - Train JSON: {'✓' if os.path.exists(train_json) else '❌ MISSING'}")
    print(f"  - Val JSON: {'✓' if os.path.exists(val_json) else '❌ MISSING'}")
    print(f"  - Train images: {'✓' if os.path.exists(train_dir) else '❌ MISSING'}")
    print(f"  - Val images: {'✓' if os.path.exists(val_dir) else '❌ MISSING'}")
    
    if all([os.path.exists(train_json), os.path.exists(val_json), 
            os.path.exists(train_dir), os.path.exists(val_dir)]):
        print("\n✅ All dataset files found! Ready to train.")
    else:
        print("\n❌ ERROR: Some dataset files are missing!")
        print("   Please check your DATASET_PATH")
else:
    print("\n❌ ERROR: Dataset path does not exist!")
    print(f"   Path: {DATASET_PATH}")
    print("\n📝 To fix:")
    print("   1. Upload your dataset to Google Drive")
    print("   2. Update DATASET_PATH in the cell above")
    print("   3. Make sure the path points to 'instance_version' folder")

print("\n" + "="*70)

## 🚀 Step 6: Start Training

In [ ]:
import subprocess
import sys

print("="*70)
print("🚀 Starting YOLO-UDD v2.0 Training")
print("="*70)

# Build training command
cmd = [
    sys.executable, 'scripts/train.py',
    '--data-dir', DATASET_PATH,
    '--batch-size', str(BATCH_SIZE),
    '--epochs', str(EPOCHS),
    '--lr', str(LEARNING_RATE),
    '--save-dir', SAVE_DIR
]

print(f"Command: {' '.join(cmd)}\n")
print("="*70)
print("Training in progress...")
print("="*70 + "\n")

# Run training
result = subprocess.run(cmd, capture_output=False, text=True)

print("\n" + "="*70)
if result.returncode == 0:
    print("✅ Training completed successfully!")
else:
    print("❌ Training ended with errors (see output above)")
print("="*70)

## 📊 Step 7: View Results

In [ ]:
import glob
import os

print("="*70)
print("📊 Training Results")
print("="*70)

checkpoint_dir = f'{SAVE_DIR}/checkpoints'

if os.path.exists(checkpoint_dir):
    checkpoints = glob.glob(f"{checkpoint_dir}/*.pt")
    
    if checkpoints:
        print(f"\n✓ Found {len(checkpoints)} checkpoint(s):")
        print("\n" + "-"*70)
        
        for ckpt in sorted(checkpoints):
            size_mb = os.path.getsize(ckpt) / (1024 * 1024)
            print(f"📦 {os.path.basename(ckpt):30s} | {size_mb:7.1f} MB")
        
        print("-"*70)
        
        # Highlight best model
        best_model = f"{checkpoint_dir}/best_model.pt"
        if os.path.exists(best_model):
            print(f"\n🏆 Best model: {best_model}")
            print("   This is the model with highest validation mAP")
        
        # Show how to download
        print("\n💾 To download checkpoints:")
        print("   from google.colab import files")
        print(f"   files.download('{best_model}')")
    else:
        print("\n⚠️  No checkpoints found")
        print("   Training may have failed - check the training output above")
else:
    print("\n⚠️  Checkpoint directory not created")
    print("   Training likely encountered an error early on")
    print("   Check the error messages in the training output above")

# Check for logs
log_dir = f'{SAVE_DIR}/logs'
if os.path.exists(log_dir):
    print(f"\n📝 Training logs saved in: {log_dir}")
    print("   View with TensorBoard:")
    print(f"   %load_ext tensorboard")
    print(f"   %tensorboard --logdir {log_dir}")

print("\n" + "="*70)

## 📥 Step 8: Download Best Model (Optional)

In [ ]:
from google.colab import files
import os

best_model = f'{SAVE_DIR}/checkpoints/best_model.pt'

if os.path.exists(best_model):
    print(f"Downloading: {best_model}")
    files.download(best_model)
    print("✓ Download started!")
else:
    print("❌ Best model not found. Make sure training completed successfully.")

## 📈 Step 9: View Training Logs (Optional)

In [ ]:
# View last 50 lines of training log
log_file = f'{SAVE_DIR}/logs/training.log'

if os.path.exists(log_file):
    !tail -50 {log_file}
else:
    print("No training log found.")

---

## 📚 Additional Resources

- **GitHub Repository:** https://github.com/kshitijkhede/YOLO-UDD-v2.0
- **Dataset:** TrashCAN 1.0
- **Paper:** YOLO-UDD v2.0 - Novel Turbidity-Adaptive Fusion for Underwater Debris Detection

---

## 🐛 Troubleshooting

### Issue: "Dataset path does not exist"
**Solution:** Update `DATASET_PATH` in Step 4 to point to your Google Drive location

### Issue: "Out of Memory (OOM)"
**Solution:** Reduce `BATCH_SIZE` to 4 or 2 in Step 4

### Issue: "No GPU detected"
**Solution:** Runtime → Change runtime type → Hardware accelerator → T4 GPU

### Issue: "Training ended with errors"
**Solution:** Check the training output for specific error messages. Common causes:
- Wrong dataset path
- Missing dataset files
- Insufficient GPU memory

---

**Made with ❤️ for Underwater Conservation** 🌊
